# R2 — Discovery & Lineage

**Deliverables produced by this notebook**  those tables live in `bronze` in this workspace):

1. **PII tags** — `pii = 'true'` on `claims.bronze.members` columns `name`, `dob`, `national_id`, `phone`, `email`.
2. **Documentation** — a comment on every `claims.gold` table (`claims_enriched`, `hospital_monthly`, `hospital_monthly_designer`) and every column of those tables.
3. **Lineage evidence** — full three-part names of all tables reading `claims.bronze.members.phone`, written to `evidence/r2_lineage_answer.md` (same folder as this notebook).

R3 (`claims.gov.audit_claims_enriched`) is intentionally out of scope for now.

In [0]:
%sql
-- R2-1: Tag the PII columns of claims.bronze.members (spec's claims.raw.members lives in bronze here)
ALTER TABLE claims.bronze.members ALTER COLUMN name SET TAGS ('pii' = 'true');
ALTER TABLE claims.bronze.members ALTER COLUMN dob SET TAGS ('pii' = 'true');
ALTER TABLE claims.bronze.members ALTER COLUMN national_id SET TAGS ('pii' = 'true');
ALTER TABLE claims.bronze.members ALTER COLUMN phone SET TAGS ('pii' = 'true');
ALTER TABLE claims.bronze.members ALTER COLUMN email SET TAGS ('pii' = 'true');

-- Verify: exactly the 5 spec columns should come back tagged pii = true
SELECT catalog_name, schema_name, table_name, column_name, tag_name, tag_value
FROM system.information_schema.column_tags
WHERE catalog_name = 'claims' AND schema_name = 'bronze' AND table_name = 'members'
ORDER BY column_name;

In [0]:
%sql
-- R2-2: Document every claims.gold table and column

-- ===== claims_enriched (31 columns) =====
COMMENT ON TABLE claims.gold.claims_enriched IS 'Gold: claim-level facts enriched with plan and hospital attributes — the primary analytics table for claims.';
COMMENT ON COLUMN claims.gold.claims_enriched.claim_id IS 'Unique claim identifier.';
COMMENT ON COLUMN claims.gold.claims_enriched.member_id IS 'Insured member who filed the claim (FK to claims.silver.members).';
COMMENT ON COLUMN claims.gold.claims_enriched.policy_id IS 'Insurance policy the claim was filed under.';
COMMENT ON COLUMN claims.gold.claims_enriched.hospital_id IS 'Hospital where treatment occurred (row-filter key for regional access).';
COMMENT ON COLUMN claims.gold.claims_enriched.admission_date IS 'Date the member was admitted.';
COMMENT ON COLUMN claims.gold.claims_enriched.discharge_date IS 'Date the member was discharged.';
COMMENT ON COLUMN claims.gold.claims_enriched.admission_month IS 'First day of the admission month (helper for monthly reporting).';
COMMENT ON COLUMN claims.gold.claims_enriched.diagnosis_code IS 'Primary diagnosis code.';
COMMENT ON COLUMN claims.gold.claims_enriched.procedure_code IS 'Primary procedure code performed.';
COMMENT ON COLUMN claims.gold.claims_enriched.is_pre_existing IS 'TRUE when the claim involves a pre-existing condition.';
COMMENT ON COLUMN claims.gold.claims_enriched.claim_type IS 'Claim category (e.g. inpatient, outpatient).';
COMMENT ON COLUMN claims.gold.claims_enriched.billed_amount IS 'Amount billed by the hospital for the claim.';
COMMENT ON COLUMN claims.gold.claims_enriched.room_rent_per_day IS 'Daily room rent charged by the hospital.';
COMMENT ON COLUMN claims.gold.claims_enriched.room_days IS 'Number of room-days in the stay.';
COMMENT ON COLUMN claims.gold.claims_enriched.status IS 'Claim outcome status (e.g. approved, rejected, pending).';
COMMENT ON COLUMN claims.gold.claims_enriched.approved_amount IS 'Amount approved for settlement.';
COMMENT ON COLUMN claims.gold.claims_enriched.rejection_code IS 'Rejection reason code; NULL when the claim was not rejected.';
COMMENT ON COLUMN claims.gold.claims_enriched.submitted_ts IS 'Timestamp the claim was submitted.';
COMMENT ON COLUMN claims.gold.claims_enriched.settled_ts IS 'Timestamp the claim was settled; NULL until settled.';
COMMENT ON COLUMN claims.gold.claims_enriched.channel IS 'Intake channel through which the claim arrived.';
COMMENT ON COLUMN claims.gold.claims_enriched.plan_code_at_admission IS 'Plan code active at the time of admission.';
COMMENT ON COLUMN claims.gold.claims_enriched.plan_family IS 'Plan family of the plan active at admission.';
COMMENT ON COLUMN claims.gold.claims_enriched.room_rent_cap_per_day IS 'Daily room-rent cap defined by the plan at admission.';
COMMENT ON COLUMN claims.gold.claims_enriched.policy_start_date IS 'Start date of the active policy.';
COMMENT ON COLUMN claims.gold.claims_enriched.policy_tenure_months IS 'Policy age in months at the time of admission.';
COMMENT ON COLUMN claims.gold.claims_enriched.length_of_stay IS 'Discharge date minus admission date, in days.';
COMMENT ON COLUMN claims.gold.claims_enriched.room_rent_over_cap IS 'TRUE when the daily room rent exceeded the plan cap.';
COMMENT ON COLUMN claims.gold.claims_enriched.turnaround_days IS 'Days from claim submission to settlement.';
COMMENT ON COLUMN claims.gold.claims_enriched.is_network_at_admission IS 'TRUE when the hospital was in the network at admission.';
COMMENT ON COLUMN claims.gold.claims_enriched.hospital_city IS 'City of the treating hospital.';
COMMENT ON COLUMN claims.gold.claims_enriched.hospital_city_tier IS 'Tier (1/2/3) of the hospital city.';

-- ===== hospital_monthly (6 columns) =====
COMMENT ON TABLE claims.gold.hospital_monthly IS 'Gold: monthly claim volume and amounts per hospital (aggregated from claims_enriched).';
COMMENT ON COLUMN claims.gold.hospital_monthly.hospital_id IS 'Hospital the monthly metrics belong to.';
COMMENT ON COLUMN claims.gold.hospital_monthly.admission_month IS 'Month of admission (first day of month).';
COMMENT ON COLUMN claims.gold.hospital_monthly.claim_count IS 'Number of claims admitted in the month.';
COMMENT ON COLUMN claims.gold.hospital_monthly.total_billed IS 'Sum of billed amounts for the month.';
COMMENT ON COLUMN claims.gold.hospital_monthly.total_approved IS 'Sum of approved amounts for the month.';
COMMENT ON COLUMN claims.gold.hospital_monthly.rejected_count IS 'Number of claims rejected in the month.';

-- ===== hospital_monthly_designer (6 columns) =====
COMMENT ON TABLE claims.gold.hospital_monthly_designer IS 'Gold (designer build): monthly claim volume and amounts per hospital — Visual-data-prep copy of hospital_monthly.';
COMMENT ON COLUMN claims.gold.hospital_monthly_designer.hospital_id IS 'Hospital the monthly metrics belong to.';
COMMENT ON COLUMN claims.gold.hospital_monthly_designer.admission_month IS 'Month of admission (first day of month).';
COMMENT ON COLUMN claims.gold.hospital_monthly_designer.claim_count IS 'Number of claims admitted in the month.';
COMMENT ON COLUMN claims.gold.hospital_monthly_designer.total_billed IS 'Sum of billed amounts for the month.';
COMMENT ON COLUMN claims.gold.hospital_monthly_designer.total_approved IS 'Sum of approved amounts for the month.';
COMMENT ON COLUMN claims.gold.hospital_monthly_designer.rejected_count IS 'Number of claims rejected in the month.';



In [0]:
%sql
-- Verify: should return ZERO rows (no undocumented user-facing table or column in gold;
-- Databricks-internal __materialization shadow tables are excluded)
SELECT *
FROM claims.information_schema.columns
WHERE  table_schema = 'gold'

## R2-3 — Lineage answer (summary)

**Tables reading `claims.bronze.members.phone`** (full three-part names, confirmed via Unity Catalog column-level lineage):

* `claims.silver.members` — pipeline streaming table (`silver.py`, pipeline `5ddb66c6-98d4-48ef-b11f-9a9b0bb4d1ea`); `phone` maps 1:1.
* `claims.gov.proof_members_admin` — R1 governance proof table (admin view of raw PII).
* `claims.gov.proof_members_masked` — R1 governance proof table (masked view of PII).

No gold table reads the phone number: `claims.gold.claims_enriched` joins members only for attributes like `member_id` — `phone` does not propagate to gold.

Full written evidence: `evidence/r2_lineage_answer.md` (same folder as this notebook).

In [0]:
# R2-3: Write the lineage evidence file (evidence/r2_lineage_answer.md, same folder as this notebook)
evidence_path = "/Workspace/Users/adikamodiya@gmail.com/Projects/saffron-shield/saffron shield-2/evidence/r2_lineage_answer.md"

content = """# R2 — Lineage Answer

**Question:** which tables read `claims.raw.members.phone`?

**Note on names:** the `claims.raw` schema holds only the landing volume in this workspace; the members table lives at **`claims.bronze.members`**. All names below are full three-part names.

**Answer (column-level lineage, Unity Catalog, confirmed 2026-11-05):**

| # | Table reading `claims.bronze.members.phone` | What it is |
| --- | --- | --- |
| 1 | `claims.silver.members` | Pipeline streaming table built by `silver.py` in pipeline `5ddb66c6-98d4-48ef-b11f-9a9b0bb4d1ea`; `bronze.members.phone` maps 1:1 to `silver.members.phone`. |
| 2 | `claims.gov.proof_members_admin` | R1 governance proof table — admin view of raw member PII (built by the R1 notebook). |
| 3 | `claims.gov.proof_members_masked` | R1 governance proof table — masked (`****<last4>`) view of member PII (built by the R1 notebook). |

## Lineage chain

```
claims.bronze.members (phone)
        |  (pipeline 5ddb66c6-98d4-48ef-b11f-9a9b0bb4d1ea, silver.py)
        v
claims.silver.members (phone)
        |  (pipeline gold.sql reads members for enrichment)
        v
claims.gold.claims_enriched  <- reads member ATTRIBUTES only; phone does NOT propagate to gold

claims.gov.proof_members_admin  (R1 notebook — direct read of bronze.members)
claims.gov.proof_members_masked (R1 notebook — direct read of bronze.members)
```

## Why no gold table appears for `phone`

`claims.gold.claims_enriched` joins members only to enrich claims with member attributes (`member_id` etc.). The `phone` column stops at `claims.silver.members` — no gold table carries it. The only other readers are the R1 proof tables, which deliberately sample `bronze.members` PII to demonstrate the `claims.gov.mask_last4` column mask.

## Verification method

Unity Catalog column-level lineage was queried for `claims.bronze.members.phone`; the downstream column set returned `claims.silver.members.phone`, `claims.gov.proof_members_admin.phone`, and `claims.gov.proof_members_masked.phone`.
"""

dbutils.fs.put(evidence_path, content, overwrite=True)
print(f"Evidence written to {evidence_path}")
print(dbutils.fs.head(evidence_path, 500))